## 1. 환경 설정 및 라이브러리 임포트

In [1]:
from dotenv import load_dotenv
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

# API 키 확인
if openai_api_key:
    print(f"✅ OpenAI API 키 로드 완료: sk-...{openai_api_key[-4:]}")
else:
    print("❌ API 키가 없습니다. .env 파일을 확인하세요.")

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ OpenAI API 키 로드 완료: sk-...yK4A


## 2. Document Upload

In [2]:
file_paths = [
    "C:/Users/user/Downloads/삼성_2025Q4_conference_eng_presentation.pdf",
    "C:/Users/user/Downloads/삼성_2025Q4_script_eng_AudioScript.pdf"
]

docs = []

for path in file_paths:
    # 각 파일을 로드
    loader = PyPDFLoader(path)
    
    # load를 통해 문서의 각 페이지를 Document 객체로 변환
    docs.extend(loader.load())

## 3. Text Splitting

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,    
    chunk_overlap=220
)

splits = text_splitter.split_documents(docs)

## 4. Embedding


In [4]:
# OpenAI 임베딩 (API 호출 방식 — GPU 불필요)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=openai_api_key
)

# Chroma 벡터스토어
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

retriever = vectorstore.as_retriever()

## 5. LLM 연결


In [5]:
chat_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=openai_api_key
)

## 6. Decomposition
**1. Query-Translation: Multi-Query**

In [6]:
# Decomposition을 위한 프롬프트 설정
template = """
You are a helpful assistant that sub-divides a complex user question into smaller, manageable sub-questions.
The target documents are written in English.
Please generate 3 sub-questions in English that will help answer the main question.

Input Query: {question}
Output (3 English sub-questions):
"""
prompt_decomposition = ChatPromptTemplate.from_template(template)


In [7]:
# Chain
generate_queries_decomposition = ( 
    prompt_decomposition 
    | chat_llm 
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

question = "삼성전자 2025 4분기 실적 발표에 대해서 알려줘."
questions = generate_queries_decomposition.invoke({"question":question})

In [9]:
for q in questions:
    print(q, "\n")

1. What are the key financial metrics that Samsung Electronics typically reports in their quarterly earnings announcements? 

2. What are the expected market conditions and trends that could impact Samsung Electronics' performance in the fourth quarter of 2025? 

3. Are there any specific products or business segments that analysts are focusing on for Samsung Electronics' Q4 2025 results? 



## 6. Decomposition
**2. Query-Translation: recursive feedback**

In [10]:
def solve_sub_queries(queries):
    """하위 질문들을 순차적으로 해결하며 맥락을 쌓아가는 함수"""
    context_accumulator = ""

    for q in queries:
        # 1. 현재 질문에 대한 관선 문서 검색
        retrieved_docs = retriever.invoke(q)
        content = "\n\n".join([d.page_content for d in retrieved_docs])

        # 2. 이전까지의 정보(context_accumulator)를 포함하여 현재 질문에 답함
        answer_prompt = f""""
        Here is some previous context: {context_accumulator}
        Current context from documents: {content}
        Question: {q}

        Provide answer to the question based on the context.
        """

        response = chat_llm.invoke(answer_prompt)
        # 3. 답변을 누적하고 다음 질문의 참고 자료로 활용
        context_accumulator += f"\n\nSub-Question: {q}\nAnswer: {response.content}"
    
    return context_accumulator


In [11]:
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are an expert AI assistant specializing in Samsung Electronics earnings reports.
Based on the following sub-questions and their answers, provide a comprehensive final response to the original question.
Answer in Korean and include specific financial figures.

#Sub-tasks and Information:
{context}

#Original Question:
{question}

#Final Answer:
""")

final_rag_chain = (
    {
        "context": lambda x: solve_sub_queries(generate_queries_decomposition.invoke(x)),
        "question": itemgetter("question")
    }
    | RAG_PROMPT
    | chat_llm
    | StrOutputParser()
)

In [12]:
from langchain_teddynote.messages import stream_response

In [ ]:
question = "삼성전자 2025 4분기 실적 발표에 대해서 알려줘."
answer = final_rag_chain.stream({"question": question})
stream_response(answer)
# langsmith trace
# https://smith.langchain.com/public/8832e596-b312-4f12-bf16-57056d587896/r

삼성전자의 2025년 4분기 실적 발표에 대한 정보는 다음과 같습니다.

삼성전자는 분기 실적 발표에서 일반적으로 다음과 같은 주요 재무 지표를 보고합니다:

1. **분기 매출**: 2025년 4분기 동안의 총 매출은 약 70조 원으로 예상되며, 이는 전년 동기 대비 감소할 것으로 보입니다.
2. **영업 이익**: 영업 이익은 약 8조 원으로 예상되며, 이는 메모리 공급 및 가격 문제로 인해 압박을 받을 것으로 보입니다.
3. **영업 이익률**: 영업 이익률은 약 11.4%로, 운영 비용을 감안한 후의 매출 비율을 나타냅니다.
4. **부문별 매출**: 디바이스 경험(DX) 부문과 디바이스 솔루션(DS) 부문으로 나누어 매출이 보고되며, 특히 모바일 디스플레이와 대형 디스플레이 부문이 주목받고 있습니다.
5. **판매 및 관리비(SG&A)**: 판매 및 관리비는 약 5조 원으로 예상되며, 이전 분기와 비교하여 증가할 것으로 보입니다.
6. **연구개발(R&D) 투자**: 연구개발 비용은 약 3조 원으로, 혁신적인 제품 개발을 위한 투자가 지속될 것입니다.
7. **환율 영향**: 환율 변동이 전체 재무 성과에 미치는 영향도 중요한 요소로, 특히 영업 이익에 영향을 미칠 것으로 예상됩니다.
8. **연간 성과**: 2025년 전체 매출은 약 300조 원으로 예상되며, 이는 전년 대비 감소할 것으로 보입니다.

2025년 4분기 삼성전자의 실적에 영향을 미칠 것으로 예상되는 시장 조건과 트렌드는 다음과 같습니다:

1. **스마트폰 수요 감소**: 계절적 요인과 메모리 공급 및 가격 문제로 인해 스마트폰 수요가 감소할 것으로 예상됩니다.
2. **메모리 공급 및 가격 문제**: 메모리 부문에서의 지속적인 문제는 비메모리 부문에도 압박을 가할 것으로 보입니다.
3. **대형 디스플레이 수요**: 연말 성수기와 관련된 대형 디스플레이 수요는 매출 성장에 긍정적인 영향을 미칠 것으로 보입니다.
4. **신제품 출시 및 혁신**: 삼성전자는 새로운 플래그십 스마트폰